#  TC contest
 


In [43]:
import re
import string

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import transformers
from transformers import AutoTokenizer
from transformers import TFRobertaModel
from datasets import load_dataset

import tensorflow_model_optimization as tfmot


## Loading the Data

In [44]:
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
train = dataset['train'].to_pandas()
test = dataset['validation'].to_pandas()

n_val = int(len(test)*0.5)
val = test[: n_val]
test = test[n_val :]

## Preprocessing

In [45]:
def remove_url(text):
    url = re.compile(r'https?://\S+|www\.\S+')
    return url.sub(r'',text)

def remove_punct(text):
    table = str.maketrans('', '', string.punctuation)
    return text.translate(table)
def remove_html(text):
    html=re.compile(r'<.*?>')
    return html.sub(r'',text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def decontraction(text):
    text = re.sub(r"won\'t", " will not", text)
    text = re.sub(r"won\'t've", " will not have", text)
    text = re.sub(r"can\'t", " can not", text)
    text = re.sub(r"don\'t", " do not", text)
    
    text = re.sub(r"can\'t've", " can not have", text)
    text = re.sub(r"ma\'am", " madam", text)
    text = re.sub(r"let\'s", " let us", text)
    text = re.sub(r"ain\'t", " am not", text)
    text = re.sub(r"shan\'t", " shall not", text)
    text = re.sub(r"sha\n't", " shall not", text)
    text = re.sub(r"o\'clock", " of the clock", text)
    text = re.sub(r"y\'all", " you all", text)

    text = re.sub(r"n\'t", " not", text)
    text = re.sub(r"n\'t've", " not have", text)
    text = re.sub(r"\'re", " are", text)
    text = re.sub(r"\'s", " is", text)
    text = re.sub(r"\'d", " would", text)
    text = re.sub(r"\'d've", " would have", text)
    text = re.sub(r"\'ll", " will", text)
    text = re.sub(r"\'ll've", " will have", text)
    text = re.sub(r"\'t", " not", text)
    text = re.sub(r"\'ve", " have", text)
    text = re.sub(r"\'m", " am", text)
    text = re.sub(r"\'re", " are", text)
    return text 

def seperate_alphanumeric(text):
    words = text
    words = re.findall(r"[^\W\d_]+|\d+", words)
    return " ".join(words)

def cont_rep_char(text):
    tchr = text.group(0) 
    
    if len(tchr) > 1:
        return tchr[0:2] 

def unique_char(rep, text):
    substitute = re.sub(r'(\w)\1+', rep, text)
    return substitute

train['text'] = train['text'].apply(lambda x : remove_url(x))
train['text'] = train['text'].apply(lambda x : remove_punct(x))
train['text'] = train['text'].apply(lambda x : remove_emoji(x))
train['text'] = train['text'].apply(lambda x : decontraction(x))
train['text'] = train['text'].apply(lambda x : seperate_alphanumeric(x))
train['text'] = train['text'].apply(lambda x : unique_char(cont_rep_char,x))

val['text'] = val['text'].apply(lambda x : remove_url(x))
val['text'] = val['text'].apply(lambda x : remove_punct(x))
val['text'] = val['text'].apply(lambda x : remove_emoji(x))
val['text'] = val['text'].apply(lambda x : decontraction(x))
val['text'] = val['text'].apply(lambda x : seperate_alphanumeric(x))
val['text'] = val['text'].apply(lambda x : unique_char(cont_rep_char,x))

test['text'] = test['text'].apply(lambda x : remove_url(x))
test['text'] = test['text'].apply(lambda x : remove_punct(x))
test['text'] = test['text'].apply(lambda x : remove_emoji(x))
test['text'] = test['text'].apply(lambda x : decontraction(x))
test['text'] = test['text'].apply(lambda x : seperate_alphanumeric(x))
test['text'] = test['text'].apply(lambda x : unique_char(cont_rep_char,x))


In [49]:
train['text'].tolist()

['BYND JPMorgan reels in expectations on Beyond Meat',
 'CCL RCL Nomura points to bookings weakness at Carnival and Royal Caribbean',
 'CX Cemex cut at Credit Suisse JP Morgan on weak building outlook',
 'ESS BTIG Research cuts to Neutral',
 'FNKO Funko slides after Piper Jaffray PT cut',
 'FTI TechnipFMC downgraded at Berenberg but called Top Pick at Deutsche Bank',
 'GM GM loses a bull',
 'GM Deutsche Bank cuts to Hold',
 'GTT Cowen cuts to Market Perform',
 'HNHAF HNHPD AAPL Trendforce cuts iPhone estimate after Foxconn delay',
 'HOG Moodys warns on HarleyDavidson',
 'HXL Citing aero ties Wells slashes PT on Hexcel',
 'I Intelsat cut to Market Perform at Raymond James',
 'KRG Compass Point cuts to Sell',
 'LK Muddy Waters goes short Luckin Coffee',
 'MANT ManTech downgraded ahead of difficult comps',
 'MDCO Oppenheimer cuts to Perform',
 'MPLX MPC MPLX cut at Credit Suisse on potential dilution from Marathon strategic review',
 'MSGN Imperial downgrades MSG Networks amid sportsfree 

In [50]:
train.to_csv("train.csv",index=False,encoding='utf-8')
val.to_csv("val.csv",index=False,encoding='utf-8')
test.to_csv("test.csv",index=False,encoding='utf-8')

## Tokenization

In [22]:
seq_len = 256
batch_size = 16
l_train = len(train)
l_val = len(val)

model_name = 'cardiffnlp/twitter-roberta-base-sentiment'

tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

x_train = tokenizer(
    train['text'].tolist(), 
    max_length=seq_len, 
    truncation=True, 
    padding='max_length', 
    add_special_tokens=True, 
    return_tensors='tf'
)

x_val = tokenizer(
    val['text'].tolist(), 
    max_length=seq_len, 
    truncation=True, 
    padding='max_length', 
    add_special_tokens=True, 
    return_tensors='tf'
)

x_test = tokenizer(
    test['text'].tolist(), 
    max_length=seq_len, 
    truncation=True, 
    padding='max_length', 
    add_special_tokens=True, 
)





y_train = train['label'].values
y_val = val['label'].values
y_test = test['label'].values

In [41]:
train['label'].values

array([0, 0, 0, ..., 2, 2, 2], dtype=int64)

In [ ]:
## Originally did one-hot encoding - I think sparse categorical makes more sense, but leaving this here just in case
# y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=3)
# y_val_oh = tf.keras.utils.to_categorical(y_val, num_classes=3)

def map_func(input_ids, attention_mask):
    return {'input_ids': input_ids, 'attention_mask': attention_mask}




train_ds = tf.data.Dataset.from_tensor_slices(
    (
        map_func(x_train['input_ids'], x_train['attention_mask']), 
        y_train
    )
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (
        map_func(x_val['input_ids'], x_val['attention_mask']), 
        y_val
    )
)


# train_ds = train_ds.shuffle(1000).batch(32)
# val_ds = val_ds.shuffle(1000).batch(32)

In [32]:
def serialize_example(feature, label):
    """Converts feature and label to tf.train.Example and serialize."""
    # Create a dictionary mapping the feature name to the tf.train.Feature
    feature = {
        'features': tf.train.Feature(float_list=tf.train.FloatList(value=tf.reshape(feature, [-1]))),
        'label': tf.train.Feature(int64_list=tf.train.Int64List(value=[label]))
    }
    # Create a Features message using tf.train.Example.
    example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
    return example_proto.SerializeToString()

def tf_serialize_example(feature, label):
    """Wrapper function to use tf.py_function for TensorFlow operations."""
    serialized_tf_string = tf.py_function(
        serialize_example,
        (feature, label),  # Pass correct tensor inputs to the serialize_example
        tf.string)         # The return type is `tf.string`
    return tf.reshape(serialized_tf_string, ())  # The result is a scalar

# Assuming `dataset` is your tf.data.Dataset with (feature, label) structure
train_ds = tf.data.Dataset.from_tensor_slices((features, labels))
train_ds_ser = train_ds.map(tf_serialize_example)

writer = tf.data.experimental.TFRecordWriter("train-ds.tfrecord")
writer.write(train_ds_ser)

NameError: name 'features' is not defined

In [7]:
map_func(x_train['input_ids'], x_train['attention_mask'])

{'input_ids': <tf.Tensor: shape=(9543, 256), dtype=int32, numpy=
 array([[    0, 19058, 13457, ...,     1,     1,     1],
        [    0,  3376,   574, ...,     1,     1,     1],
        [    0,   347,  1000, ...,     1,     1,     1],
        ...,
        [    0,   705,   565, ...,     1,     1,     1],
        [    0, 45213, 30065, ...,     1,     1,     1],
        [    0,   975, 13457, ...,     1,     1,     1]])>,
 'attention_mask': <tf.Tensor: shape=(9543, 256), dtype=int32, numpy=
 array([[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0]])>}

In [ ]:

test_ds = tf.data.Dataset.from_tensor_slices(
    (
        map_func(x_test['input_ids'], x_test['attention_mask']), 
        y_test
    )
)

test_ds = test_ds.shuffle(1000).batch(32)


In [27]:
train_ds

<TensorSliceDataset element_spec=({'input_ids': TensorSpec(shape=(256,), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(256,), dtype=tf.int32, name=None)}, TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [28]:
val_ds.save("val-ds")

In [29]:
load_dataset("val-ds")

DataFilesNotFoundError: No (supported) data files found in val-ds